# NB5_a_Baselines_Colab_TestRun
HF project: `boods/FrMedQA-CrossLingual-v2-PPL-<qlora|bf16>-*`
Path: `/content/drive/MyDrive/00_PHD_THESIS/frmedqa-colab-v2`


In [ ]:
# ⚙️ Stage 0 — Install dependencies (Colab)
import subprocess, sys, importlib

REQUIRED = {"unsloth":"unsloth","transformers":"transformers","peft":"peft",
            "bitsandbytes":"bitsandbytes","datasets":"datasets",
            "rouge_score":"rouge-score","nltk":"nltk","bert_score":"bert-score",
            "huggingface_hub":"huggingface-hub","tqdm":"tqdm",
            "sklearn":"scikit-learn","sacrebleu":"sacrebleu", "modelscope": "modelscope",
            "datasketch": "datasketch"}

missing_pkgs_to_install = []
for mod, pkg in REQUIRED.items():
    if importlib.util.find_spec(mod) is None:
        missing_pkgs_to_install.append(pkg)

if missing_pkgs_to_install:
    print(f"Attempting to install missing packages: {missing_pkgs_to_install}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs_to_install)
        print("Successfully installed missing packages. A kernel restart is recommended for full effect.")
    except subprocess.CalledProcessError as e:
        print(f"Error during installation of {missing_pkgs_to_install}: {e}. Please install manually.")
        raise
    except Exception as e:
        print(f"An unexpected error occurred during package installation: {e}")
        raise

try:
    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)
except Exception:
    pass
print("✓ Stage 0 done")


In [ ]:
# ⚙️ Stage 0b — Drive mount, paths, secrets
import os, sys

def _on_colab():
    try:
        import google.colab; return True
    except Exception: return False

if _on_colab():
    from google.colab import drive
    drive.mount("/content/drive")

    def _secret(name, prompt):
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v: return v
        except Exception: pass
        import getpass
        return getpass.getpass(prompt)
    os.environ["HF_TOKEN"] = _secret("HF_TOKEN", "HF_TOKEN: ")
    _wb = _secret("WANDB_API_KEY", "WANDB_API_KEY (blank to skip): ")
    if _wb:
        os.environ["WANDB_API_KEY"] = _wb
    else:
        os.environ["WANDB_DISABLED"] = "true"

BASE_DIR = os.environ.get("FRMEDQA_BASE", "/content/drive/MyDrive/00_PHD_THESIS/frmedqa-colab-v2")
REPO_DIR    = BASE_DIR
RESULTS_DIR = os.path.join(BASE_DIR, "RESULT")
EVALS_DIR   = os.path.join(BASE_DIR, "EVALS")
CACHE_DIR   = os.path.join(EVALS_DIR, "results_cache")
OUT_DIR     = os.path.join(BASE_DIR, "analysis_output")
FIG_DIR     = os.path.join(OUT_DIR, "figures")
for d in (RESULTS_DIR, EVALS_DIR, CACHE_DIR, OUT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["FRMEDQA_REPO"] = REPO_DIR
os.environ["FRMEDQA_BASE"] = BASE_DIR
os.environ["FRMEDQA_HF_USER"] = "boods"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for v in ("HF_HOME", "HF_HUB_CACHE", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE"):
    os.environ[v] = os.path.join(BASE_DIR, "hf_cache")

print(f"BASE_DIR: {BASE_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")


In [ ]:
os.environ["FRMEDQA_PROJECT_NAME"] = "FrMedQA-CrossLingual-v2-PPL"
if os.environ.get("FRMEDQA_SEED"):   # cluster multi-seed run: one HF namespace per seed
    os.environ["FRMEDQA_PROJECT_NAME"] += f"-s{os.environ['FRMEDQA_SEED']}"
print(f"HF project: {os.environ['FRMEDQA_PROJECT_NAME']}")

# ─── ONE precision for the whole project (single source of truth) ────────
# qlora = 4-bit NF4 frozen base + bf16 LoRA (QLoRA)   | fits L4 22 GB / A100 40 GB
# bf16  = bf16 frozen base + bf16 LoRA (16-bit LoRA) | needs A100 80 GB / H100
# To switch, edit PRECISION.txt on Drive; every notebook reads the same file.
_pf = os.path.join(BASE_DIR, "PRECISION.txt")
if not os.path.exists(_pf):
    open(_pf, "w").write("qlora")
os.environ["FRMEDQA_PRECISION"] = open(_pf).read().strip().lower()
assert os.environ["FRMEDQA_PRECISION"] in ("qlora", "bf16"), "PRECISION.txt must contain qlora or bf16"
print(f"Precision: {os.environ['FRMEDQA_PRECISION']}  (from {_pf})")


# NB5a — Vanilla Baselines (Colab Test Run)
**local Unsloth** (small models that fit in Colab GPU).
Run the **Colab bootstrap** cell first. Add `OPENROUTER_API_KEY` in Colab Secrets.

# 📓 NB5_a — Vanilla Baseline Evaluation
**EnToFrMedicaLLM pipeline · Notebook 5a of 6**

Evaluates the **vanilla baseline models** (no adapters):
- Qwen3-14B-vanilla
- Mistral-7B-Instruct-v0.3
- Llama-2-13B

Each model is evaluated on every task (MCQA, ExtQA, AbsQA) at 0/3/5-shot.
Results are cached per (model × task × shot) under `EVAL_CACHE_DIR` so this
notebook can be interrupted and resumed at any point.

Run **NB5_b** afterwards to evaluate the EnMed adapters; both notebooks
write into the same cache directory and the final summary in NB5_b reads
from both.

## ⚙️ Stage 0 — Install

In [ ]:
# Stage 0++ — Use ModelScope mirror for Unsloth model downloads
#
# Hugging Face downloads from `unsloth/*` repos can be slow or rate-limited
# in some regions. Setting UNSLOTH_USE_MODELSCOPE=1 routes downloads through
# ModelScope, which mirrors Unsloth's models and is typically much faster.
# This must be set BEFORE any `import unsloth` or `from unsloth import ...`.
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
print("✓ ModelScope mirror enabled for Unsloth")

✓ ModelScope mirror enabled for Unsloth


In [ ]:
# Stage 0+ — Suppress noisy deprecation/info messages
#
# Eval prints at length per-test-row across many models × shots × tasks.
# We silence three known sources of spam that don't affect correctness:
#   1. transformers AttentionMaskConverter deprecation (per-layer, per-forward)
#   2. Unsloth "Restored added_tokens_decoder metadata" (per checkpoint, per
#      adapter merge — applies here whenever we PEFT-load + merge adapters)
#   3. Generic transformers `max_new_tokens` / `max_length` reminder
import warnings, logging

# (1) AttentionMaskConverter deprecation chatter
for _msg in [r".*attention mask API.*",
             r".*AttentionMaskConverter.*",
             r".*modeling_attn_mask_utils.*"]:
    warnings.filterwarnings("ignore", message=_msg, category=FutureWarning)

for _name in ["transformers.modeling_attn_mask_utils",
              "transformers.masking_utils"]:
    logging.getLogger(_name).setLevel(logging.ERROR)

# (2) Unsloth chatter — INFO-level by default, raise to WARNING
for _name in ["unsloth", "unsloth.save", "unsloth.tokenizer_utils",
              "unsloth.chat_templates"]:
    logging.getLogger(_name).setLevel(logging.WARNING)

# (3) max_new_tokens vs max_length reminder
warnings.filterwarnings("ignore",
    message=r".*max_new_tokens.*max_length.*",
    category=UserWarning)

# Some Unsloth versions print directly with `print()` rather than logging.
# Monkey-patch builtin print to swallow the specific noisy lines.
import builtins as _bi
_real_print = _bi.print
def _filtered_print(*args, **kwargs):
    if args and isinstance(args[0], str) and (
        "Restored added_tokens_decoder metadata" in args[0]
        or "attention mask API" in args[0]
    ):
        return  # swallow
    return _real_print(*args, **kwargs)
_bi.print = _filtered_print

print("✓ Stage 0+ done — warning suppressors active")

✓ Stage 0+ done — warning suppressors active


## 🔑 Stage 1 — Boilerplate / config / auth

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Stage 1 — import shared modules (cluster-ready)
#   Code (enmed_core / enmed_recipe) comes from the cloned repo + `poetry install`.
#   DATA & OUTPUTS live under $FRMEDQA_BASE (point it at scratch). Code != data.
import os, sys
BASE_DIR = os.environ.get("FRMEDQA_BASE", os.path.expanduser("~/frmedqa_experiments"))
def _find_repo_on_path():
    try:
        import enmed_core  # already importable (cwd = repo root, or installed)
        return
    except ModuleNotFoundError:
        for cand in [os.environ.get("FRMEDQA_REPO"), os.getcwd(),
                     os.path.abspath(os.path.join(os.getcwd(), ".."))]:
            if cand and os.path.exists(os.path.join(cand, "enmed_core.py")):
                sys.path.insert(0, cand); return
        raise ModuleNotFoundError("enmed_core not found — run `poetry install` in the repo, launch Jupyter from the repo root, or set $FRMEDQA_REPO to the repo path.")
_find_repo_on_path()
import enmed_core as ec
print(f"✓ enmed_core imported | BASE_DIR = {BASE_DIR}")

Mounted at /content/drive
✓ enmed_core imported


In [ ]:
# Stage 1b — Config / logger / auth
cfg = ec.EnMedConfig(
    base_dir=BASE_DIR, seed=42,
    eval_n_shots=[0, 3, 5],
    eval_max_new_tokens_mcqa=64,     # thinking suppressed; 64 is generous for letters
    eval_max_new_tokens_extqa=64,    # cut hard: extracted spans must be SHORT
    eval_max_new_tokens_absqa=512,
    eval_temperature=0.0,
    eval_subset_size=None,   # None = full test set; set int for debug
)
cfg.make_dirs()
logger = ec.setup_logger("nb5",
    log_file=os.path.join(cfg.evals_dir, f"nb5_eval_{ec.now_ts()}.log"))
ec.seed_all(cfg.seed)
ec.colab_keep_alive()

HF_TOKEN = ec.bootstrap_hf(logger=logger)

EVAL_CACHE_DIR = os.path.join(cfg.evals_dir, "results_cache")
os.makedirs(EVAL_CACHE_DIR, exist_ok=True)
logger.info(f"Eval cache dir: {EVAL_CACHE_DIR}")

<IPython.core.display.Javascript object>

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


20:09:23 | I | ✓ HF authenticated
20:09:23 | I | Eval cache dir: /content/drive/MyDrive/00_PHD_THESIS/CODE/05-05-26/EVALS/results_cache


## 📚 Stage 2 — Load test set + exemplar pool

In [ ]:
# Stage 2 — Read NB1 outputs
TEST_PATH = os.path.join(cfg.results_dir, "test.jsonl")
EXEM_PATH = os.path.join(cfg.results_dir, "exemplars.jsonl")
for p in (TEST_PATH, EXEM_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"NB1 output missing: {p} — run NB1 first.")

test_rows  = ec.load_jsonl(TEST_PATH)
exem_rows  = ec.load_jsonl(EXEM_PATH)

if cfg.eval_subset_size:
    import random as _r
    _r.Random(cfg.seed).shuffle(test_rows)
    test_rows = test_rows[:cfg.eval_subset_size]
    logger.warning(f"DEBUG MODE — capped test to {len(test_rows)}")

def _by_task(rows, t): return [r for r in rows if r["task"] == t]
TEST = {t: _by_task(test_rows, t) for t in ["mcqa","extqa","absqa"]}
POOL = {t: _by_task(exem_rows, t) for t in ["mcqa","extqa","absqa"]}
logger.info(f"  TEST  | mcqa={len(TEST['mcqa']):,}  extqa={len(TEST['extqa']):,}  absqa={len(TEST['absqa']):,}")
logger.info(f"  POOL  | mcqa={len(POOL['mcqa']):,}  extqa={len(POOL['extqa']):,}  absqa={len(POOL['absqa']):,}")

20:09:35 | I |   TEST  | mcqa=622  extqa=207  absqa=248
20:09:35 | I |   POOL  | mcqa=200  extqa=200  absqa=200


## 🤖 Stage 3 — Model registry (baselines + ours)

In [ ]:
# Stage 3 — Model registry: the vanilla reference.
# Only unsloth/Qwen3-14B, evaluated locally at cfg.precision.

REGISTRY = [
    # The ONLY vanilla model: the Unsloth base checkpoint unsloth/Qwen3-14B
    # (cfg.base_model), run locally at cfg.precision exactly like the
    # Perplexity-* / NoPPL-* models, so precision is not a confound.
    {"name": "Qwen3-14B-vanilla", "base": cfg.base_model, "adapters": []},
]
assert cfg.base_model == "unsloth/Qwen3-14B", cfg.base_model

for e in REGISTRY:
    logger.info(f"  {e['name']:30s}  [local, {cfg.precision}] {e['base']}")


20:09:38 | I |   Qwen3-14B-vanilla               unsloth/Qwen3-14B-unsloth-bnb-4bit
20:09:38 | I |   Qwen3-8B                        unsloth/Qwen3-8B-unsloth-bnb-4bit
20:09:38 | I |   Mistral-7B-Instruct-v0.3        unsloth/mistral-7b-instruct-v0.3-bnb-4bit
20:09:38 | I |   Llama-2-13B                     unsloth/llama-2-13b-bnb-4bit


## 🔮 Stage 4 — Inference engine (Unsloth)

In [ ]:
# Stage 4 — Inference engine. Loads base in 4-bit, then overlays adapters in
# order via PEFT, merging each before the next so we end with a single merged
# 4-bit model ready for fast inference.
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
import torch, warnings, gc
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

warnings.filterwarnings("ignore",
    message=r".*max_new_tokens.*max_length.*",
    category=UserWarning)


def _hard_vram_reset():
    """Aggressively free VRAM between model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass
    gc.collect()


class UnslothEngine:
    def __init__(self, base, adapters=None, max_seq=2048):
        from peft import PeftModel
        _hard_vram_reset()

        self.base = base
        self.adapters = list(adapters or [])

        self.model, self.tokenizer = ec.load_model(cfg, base, max_seq_length=max_seq,
                                                   logger=logger)

        # At most ONE adapter (the task LoRA) on top of the merged DAPT base.
        # It stays UNMERGED: merging into 4-bit weights would add rounding
        # error the adapter never saw during training.
        if len(self.adapters) > 1:
            raise ValueError(f"expected at most 1 adapter, got {self.adapters}")
        for adapter_repo in self.adapters:
            logger.info(f"    attaching adapter (unmerged): {adapter_repo}")
            self.model = PeftModel.from_pretrained(self.model, adapter_repo, token=HF_TOKEN)
            self.model.eval()
            _hard_vram_reset()

        try:
            self.tokenizer = get_chat_template(self.tokenizer,
                                               chat_template="qwen3")
        except Exception:
            pass
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        try:
            FastLanguageModel.for_inference(self.model)
        except Exception as _e:
            logger.warning(f"    for_inference skipped: {_e}")

        # Strip max_length from default GenerationConfig — avoids the
        # "Both max_new_tokens and max_length set" warning spam.
        if hasattr(self.model, "generation_config") and self.model.generation_config is not None:
            self.model.generation_config.max_length = None
            self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
            if self.model.generation_config.eos_token_id is None:
                self.model.generation_config.eos_token_id = self.tokenizer.eos_token_id

    @torch.inference_mode()
    def generate(self, messages, max_new_tokens=256, temperature=0.0):
        """Generate with thinking SUPPRESSED at three levels.

        Qwen3 was pretrained with thinking enabled, so even with the plain
        `qwen3` chat template it tends to emit a <think>...</think> block
        before the answer. With small token budgets (e.g. MCQA=32) the
        model exhausts its budget mid-thinking and `pred` ends up empty.

        We force the model to skip thinking via three layers of defense:
          1. Pass `enable_thinking=False` to apply_chat_template — Qwen3's
             tokenizer recognizes this kwarg and inserts an empty thinking
             block in the prompt header, signaling "thinking already done".
          2. Append literal "<think></think>\n\n" to the prompt as a
             mechanical fallback for tokenizers that ignore (1).
          3. Strip any residual <think>...</think> from the output as a
             final safety net (defense in depth).
        """
        # Layer 1 + 2: build prompt with thinking suppressed
        try:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,            # ← Qwen3 specific
            )
        except TypeError:
            # Older tokenizer doesn't accept enable_thinking — fall back
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            prompt = "\n".join(f"{m['role']}: {m['content']}"
                                for m in messages) + "\nassistant:"

        # Layer 2 (mechanical fallback): if the prompt doesn't already end
        # with a closed think block, append one. Qwen3 will then continue
        # generating *after* it, producing the answer directly.
        if "<think></think>" not in prompt and "<think>\n\n</think>" not in prompt:
            prompt = prompt.rstrip() + "<think></think>\n\n"

        inputs = self.tokenizer(prompt, return_tensors="pt",
                                return_attention_mask=True).to(self.model.device)
        out = self.model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            max_length=None,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9 if temperature > 0 else 1.0,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        gen = out[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(gen, skip_special_tokens=True)

        # Layer 3 (safety net): strip any <think>...</think> that leaked
        # through, plus dangling <think> from truncated outputs.
        import re as _re
        text = _re.sub(r"<think>.*?</think>", "", text, flags=_re.DOTALL)
        text = _re.sub(r"<think>.*",          "", text, flags=_re.DOTALL)
        return text.strip()

    def close(self):
        # Important: explicit del + reset, not just `del self.model`,
        # because PEFT/Unsloth keep weak references that hold VRAM.
        try:
            self.model.cpu()
        except Exception:
            pass
        del self.model
        del self.tokenizer
        _hard_vram_reset()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


### One-time cache cleanup after applying the Qwen3 fix
Run this once, then re-run Stage 6 to regenerate Qwen3 outputs.

In [ ]:
# One-time cleanup for the v2 pipeline (precision-correct local Qwen3-14B-vanilla).
# Old cache files came from the previous, flawed pipeline and have the same
# file names, so they would be treated as "already done". This cell deletes
# them ONCE, then writes a marker so re-running it never deletes new results.
import glob
_marker = os.path.join(EVAL_CACHE_DIR, ".v2_vanilla_{cfg.precision}_cleaned".format(cfg=cfg))
if os.path.exists(_marker):
    logger.info(f"  v2 cleanup already done ({_marker}) — nothing deleted")
else:
    _n = 0
    for _pat in ['Qwen3-14B-vanilla__*.json']:
        for _fp in glob.glob(os.path.join(EVAL_CACHE_DIR, _pat)):
            os.remove(_fp); _n += 1
    open(_marker, "w").write("done")
    logger.info(f"  v2 cleanup: removed {_n} old cache file(s); marker written")


## 🎯 Stage 5 — Per-task runners

In [ ]:
# Stage 5a — MCQA runner
import re
from tqdm.auto import tqdm

# Local cleaning helpers (kept here so this cell is self-contained for runners).
_RUN_THINK_RE       = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)
_RUN_DANGLING_THINK = re.compile(r"<think>.*",          flags=re.DOTALL | re.IGNORECASE)
_RUN_TAG_RE         = re.compile(r"</?(?:think|reasoning|reflection|analysis)\b[^>]*>",
                                 flags=re.IGNORECASE)
_RUN_IM_TOKEN       = re.compile(r"<\|im_(?:start|end)\|>(?:assistant|user|system)?")

def _clean_gen(text):
    """Strip <think>, dangling <think>, and chat control tokens."""
    if not text: return ""
    s = _RUN_THINK_RE.sub("", text)
    s = _RUN_DANGLING_THINK.sub("", s)
    s = _RUN_TAG_RE.sub("", s)
    s = _RUN_IM_TOKEN.sub("", s)
    return s.strip()


def _opts(opts):
    if isinstance(opts, dict): return opts
    if isinstance(opts, list): return {chr(65+i): str(o) for i,o in enumerate(opts)}
    return {}


def run_mcqa(engine, test, pool, n_shot, model_name, seed=42):
    out_rows = []
    for ex in tqdm(test, desc=f"  MCQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"question": d["question"],
                  "options":  _opts(d["options"]),
                  "answer":   d["answer"]} for d in demos_raw]
        msgs = ec.build_mcqa_messages(ex["question"], _opts(ex["options"]), demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_mcqa,
                              temperature=cfg.eval_temperature)
        # Strip <think> from raw before parsing letters
        cleaned = _clean_gen(gen)
        n_choices = len(_opts(ex["options"])) or 5
        pred_letters = ec.parse_mcqa_letters(cleaned, n_choices=n_choices)
        gold_letters = sorted(set(c for c in (ex["answer"] or "").split(",")
                                  if c.strip().isalpha()))
        out_rows.append({
            "id": ex["id"], "n_shot": n_shot,
            "raw":  gen, "cleaned": cleaned,
            "pred": ",".join(pred_letters),
            "gold": ",".join(gold_letters),
            "n_choices": n_choices,
        })
    return out_rows


def score_mcqa(rows):
    """Letter-set accuracy + macro F1 + Hamming over A..Z."""
    import numpy as np
    if not rows: return {}
    set_acc = np.mean([set(r["pred"].split(",")) == set(r["gold"].split(","))
                       for r in rows])
    n_choices = max(r["n_choices"] for r in rows)
    f1s = []
    for i in range(n_choices):
        L = chr(65+i)
        yt = [1 if L in r["gold"].split(",") else 0 for r in rows]
        yp = [1 if L in r["pred"].split(",") else 0 for r in rows]
        if not any(yt) and not any(yp): continue
        tp = sum(t and p for t,p in zip(yt,yp))
        fp = sum((not t) and p for t,p in zip(yt,yp))
        fn = sum(t and (not p) for t,p in zip(yt,yp))
        if tp+fp == 0 or tp+fn == 0: f1s.append(0.0); continue
        prec = tp/(tp+fp); rec = tp/(tp+fn)
        f1s.append(0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec))
    macro_f1 = float(np.mean(f1s)) if f1s else 0.0
    ham = np.mean([ec.hamming_accuracy(r["pred"].split(",") if r["pred"] else [],
                                       r["gold"].split(",") if r["gold"] else [],
                                       n_choices=r["n_choices"]) for r in rows])
    return {"accuracy": float(set_acc), "macro_f1": macro_f1,
            "hamming":  float(ham), "n": len(rows)}

In [ ]:
# Stage 5b — ExtQA runner
def _trim_to_short_span(text, max_words=20):
    """ExtQA spans should be short. Take the first non-empty line and cap word count."""
    if not text: return ""
    s = _clean_gen(text)
    # First non-empty line
    for line in s.splitlines():
        line = line.strip().strip('"').strip("«»").strip("'")
        if line:
            words = line.split()
            return " ".join(words[:max_words])
    return s


def run_extqa(engine, test, pool, n_shot, model_name, seed=42):
    out = []
    for ex in tqdm(test, desc=f"  ExtQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"context": d.get("context",""),
                  "question": d["question"], "answer": d["answer"]}
                 for d in demos_raw]
        msgs = ec.build_extqa_messages(ex.get("context",""), ex["question"], demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_extqa,
                              temperature=cfg.eval_temperature)
        pred = _trim_to_short_span(gen)
        out.append({"id": ex["id"], "n_shot": n_shot,
                    "raw": gen, "pred": pred, "gold": ex["answer"]})
    return out


def score_extqa(rows):
    if not rows: return {}
    import numpy as np
    em  = np.mean([ec.exact_match(r["pred"], r["gold"]) for r in rows])
    f1  = np.mean([ec.token_f1   (r["pred"], r["gold"]) for r in rows])
    return {"em": float(em), "token_f1": float(f1), "n": len(rows)}

In [ ]:
# Stage 5c — AbsQA runner
def run_absqa(engine, test, pool, n_shot, model_name, seed=42):
    out = []
    for ex in tqdm(test, desc=f"  AbsQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"context": d.get("context"),
                  "question": d["question"], "answer": d["answer"]}
                 for d in demos_raw]
        msgs = ec.build_absqa_messages(ex["question"], context=ex.get("context"), demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_absqa,
                              temperature=cfg.eval_temperature)
        pred = _clean_gen(gen)
        out.append({"id": ex["id"], "n_shot": n_shot,
                    "raw": gen, "pred": pred, "gold": ex["answer"]})
    return out


def score_absqa(rows, bertscore_model="almanach/camembert-bio-base"):
    if not rows: return {}
    import numpy as np
    rouge = np.mean([ec.compute_rouge_l(r["pred"], r["gold"]) for r in rows])
    bleu  = np.mean([ec.compute_bleu4 (r["pred"], r["gold"]) for r in rows])
    out = {"rouge_l": float(rouge), "bleu4": float(bleu), "n": len(rows)}
    try:
        preds = [r["pred"] for r in rows]
        golds = [r["gold"] for r in rows]
        _, _, F1 = ec.compute_bertscore_batch(preds, golds, model_name=bertscore_model)
        out["bertscore_f1"] = float(np.mean(F1))
    except Exception as e:
        logger.warning(f"  BERTScore failed: {e}")
        out["bertscore_f1"] = None
    return out

## 🚀 Stage 6 — Evaluation loop (crash-safe, cached)

In [ ]:
# Stage 6 — Evaluate every model × every task × every shot count
def cache_path(model_name, task, n_shot):
    safe = model_name.replace("/","_")
    return os.path.join(EVAL_CACHE_DIR, f"{safe}__{task}__{n_shot}shot.json")


def run_all_for(entry):
    """Evaluate one model entry across all (task, n_shot) combos with caching."""
    model_name = entry["name"]
    needed = []
    for t in ["mcqa","extqa","absqa"]:
        for s in cfg.eval_n_shots:
            if not os.path.exists(cache_path(model_name, t, s)):
                needed.append((t, s))
    if not needed:
        logger.info(f"  ↺ {model_name} fully cached")
        return

    # Pre-load VRAM scrub. The previous engine's close() also called this,
    # but a second pass after gc finalizers is essential on Colab L4.
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    free_gib = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
    logger.info(f"▶ Loading {model_name} | free_VRAM={free_gib:.1f} GiB | "
                f"base={entry['base']} | precision={cfg.precision}")
    engine = UnslothEngine(
        base=entry["base"],
        adapters=entry.get("adapters", []),
        max_seq=cfg.max_seq_length,
    )

    try:
        for task, shot in needed:
            try:
                if task == "mcqa":
                    rows = run_mcqa(engine, TEST["mcqa"], POOL["mcqa"], shot, model_name)
                    metrics = score_mcqa(rows)
                elif task == "extqa":
                    rows = run_extqa(engine, TEST["extqa"], POOL["extqa"], shot, model_name)
                    metrics = score_extqa(rows)
                else:
                    rows = run_absqa(engine, TEST["absqa"], POOL["absqa"], shot, model_name)
                    metrics = score_absqa(rows)
                payload = {"model": model_name,
                           "base": entry.get("base", entry.get("model_id", "")),
                           "adapters": entry.get("adapters", []),
                           "task": task, "n_shot": shot,
                           "metrics": metrics, "rows": rows,
                           "ts": ec.now_ts()}
                ec.atomic_write_json(payload, cache_path(model_name, task, shot))
                logger.info(f"  ✓ {model_name} | {task} | {shot}-shot | {metrics}")
            except Exception as e:
                logger.error(f"  ✗ {model_name} | {task} | {shot}-shot: {e}")
    finally:
        engine.close()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.ipc_collect()

In [ ]:
# Stage 6b — Run the loop
import traceback
for entry in REGISTRY:
    try:
        run_all_for(entry)
    except Exception as e:
        logger.error(f"Model {entry['name']} failed entirely: {e}")
        traceback.print_exc()
        ec.cleanup()

20:10:21 | I | ▶ Loading Qwen3-14B-vanilla | free_VRAM=21.8 GiB | base=unsloth/Qwen3-14B-unsloth-bnb-4bit | adapters=[]


2026-05-06 20:10:27,630 - modelscope - INFO - Got 15 files, start to download ...


Processing 15 items:   0%|          | 0.00/15.0 [00:00<?, ?it/s]

2026-05-06 21:55:56,291 - modelscope - INFO - Finish downloading 15 files for repo 'unsloth/Qwen3-14B-unsloth-bnb-4bit'


==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

12:00:45 | W |   BERTScore failed: 'almanach/camembert-bio-base'
12:00:45 | I |   ✓ Llama-2-13B | absqa | 0-shot | {'rouge_l': 0.04342475451196912, 'bleu4': 0.0025916565401018876, 'n': 248, 'bertscore_f1': None}


  AbsQA Llama-2-13B 3-shot:   0%|          | 0/248 [00:00<?, ?it/s]

14:00:29 | W |   BERTScore failed: 'almanach/camembert-bio-base'
14:00:29 | I |   ✓ Llama-2-13B | absqa | 3-shot | {'rouge_l': 0.008892788045009457, 'bleu4': 0.0003442553657134626, 'n': 248, 'bertscore_f1': None}


  AbsQA Llama-2-13B 5-shot:   0%|          | 0/248 [00:00<?, ?it/s]

16:00:55 | W |   BERTScore failed: 'almanach/camembert-bio-base'
16:00:55 | I |   ✓ Llama-2-13B | absqa | 5-shot | {'rouge_l': 0.045592703716912576, 'bleu4': 0.0018495675307597336, 'n': 248, 'bertscore_f1': None}


## 🏁 Stage 7 — Done. Open NB5_b to evaluate EnMed adapters.

Cached results from this notebook live in `EVAL_CACHE_DIR` and will be
combined with NB5_b's outputs in the unified summary table.

In [ ]:
print("═" * 70)
print(f"  NB5_a (BASELINES) COMPLETE — {ec.now_ts()}")
print("═" * 70)
print(f"  Cached results: {EVAL_CACHE_DIR}")
print("═" * 70)
print("  → Open NB5_b_Evaluation.ipynb to evaluate EnMed adapters")

══════════════════════════════════════════════════════════════════════
  NB5_a (BASELINES) COMPLETE — 20260507_160101
══════════════════════════════════════════════════════════════════════
  Cached results: /content/drive/MyDrive/00_PHD_THESIS/CODE/05-05-26/EVALS/results_cache
══════════════════════════════════════════════════════════════════════
  → Open NB5_b_Evaluation.ipynb to evaluate EnMed adapters
